# 🎧 Bluetooth Headphones Price Prediction - EDA & Data Preparation

This notebook performs exploratory data analysis on the scraped headphones dataset and prepares it for model training.

## Objectives
1. Load and explore the enhanced dataset
2. Visualize price distributions and relationships
3. Handle missing values and duplicates
4. Engineer new features
5. Prepare train-test splits
6. Save processed dataset

In [ ]:
# Install required packages (uncomment if running in Google Colab)
# !pip install pandas numpy matplotlib seaborn scikit-learn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Load Dataset

In [ ]:
# Load the enhanced dataset
df = pd.read_csv('../data/enhanced-headphones-dataset.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
df.head()

## 2. Data Overview & Summary Statistics

In [ ]:
# Basic info
print("=" * 80)
print("DATASET INFORMATION")
print("=" * 80)
df.info()

In [ ]:
# Summary statistics
print("\n" + "=" * 80)
print("NUMERICAL FEATURES SUMMARY")
print("=" * 80)
df.describe()

In [ ]:
# Missing values analysis
print("\n" + "=" * 80)
print("MISSING VALUES ANALYSIS")
print("=" * 80)
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Percentage': missing_pct
}).sort_values('Percentage', ascending=False)
print(missing_df[missing_df['Missing Count'] > 0])

In [ ]:
# Visualize missing values
plt.figure(figsize=(14, 6))
missing_data = df.isnull().sum()
missing_data = missing_data[missing_data > 0].sort_values(ascending=False)
sns.barplot(x=missing_data.values, y=missing_data.index, palette='viridis')
plt.title('Missing Values by Feature', fontsize=16, fontweight='bold')
plt.xlabel('Count of Missing Values', fontsize=12)
plt.ylabel('Features', fontsize=12)
plt.tight_layout()
plt.show()

## 3. Price Distribution Analysis

In [ ]:
# Price distribution
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Histogram
axes[0].hist(df['price_inr'].dropna(), bins=50, color='skyblue', edgecolor='black')
axes[0].set_title('Price Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Price (INR)', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)
axes[0].axvline(df['price_inr'].median(), color='red', linestyle='--', label=f'Median: ₹{df["price_inr"].median():.0f}')
axes[0].legend()

# Box plot
axes[1].boxplot(df['price_inr'].dropna(), vert=True)
axes[1].set_title('Price Box Plot', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Price (INR)', fontsize=12)

plt.tight_layout()
plt.show()

print(f"\nPrice Statistics:")
print(f"  Mean: ₹{df['price_inr'].mean():.2f}")
print(f"  Median: ₹{df['price_inr'].median():.2f}")
print(f"  Min: ₹{df['price_inr'].min():.2f}")
print(f"  Max: ₹{df['price_inr'].max():.2f}")
print(f"  Std Dev: ₹{df['price_inr'].std():.2f}")

## 4. Category Analysis

In [ ]:
# Category distribution
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Count plot
category_counts = df['category'].value_counts()
axes[0].bar(category_counts.index, category_counts.values, color=['#FF6B6B', '#4ECDC4', '#45B7D1'])
axes[0].set_title('Product Count by Category', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Category', fontsize=12)
axes[0].set_ylabel('Count', fontsize=12)
axes[0].tick_params(axis='x', rotation=45)

# Price by category
df.boxplot(column='price_inr', by='category', ax=axes[1])
axes[1].set_title('Price Distribution by Category', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Category', fontsize=12)
axes[1].set_ylabel('Price (INR)', fontsize=12)
plt.suptitle('')

plt.tight_layout()
plt.show()

print("\nCategory Statistics:")
print(df.groupby('category')['price_inr'].agg(['count', 'mean', 'median', 'min', 'max']))

## 5. Brand Analysis

In [ ]:
# Top brands by count
top_brands = df['brand'].value_counts().head(15)

plt.figure(figsize=(14, 6))
sns.barplot(x=top_brands.values, y=top_brands.index, palette='coolwarm')
plt.title('Top 15 Brands by Product Count', fontsize=16, fontweight='bold')
plt.xlabel('Number of Products', fontsize=12)
plt.ylabel('Brand', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Average price by top brands
brand_price = df.groupby('brand')['price_inr'].agg(['mean', 'count']).sort_values('mean', ascending=False)
brand_price_top = brand_price[brand_price['count'] >= 3].head(15)

plt.figure(figsize=(14, 6))
sns.barplot(x=brand_price_top['mean'], y=brand_price_top.index, palette='viridis')
plt.title('Average Price by Brand (min 3 products)', fontsize=16, fontweight='bold')
plt.xlabel('Average Price (INR)', fontsize=12)
plt.ylabel('Brand', fontsize=12)
plt.tight_layout()
plt.show()

## 6. Feature Relationships

In [ ]:
# Price vs Battery Life
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Scatter plot
axes[0].scatter(df['battery_life_hrs'], df['price_inr'], alpha=0.5, color='teal')
axes[0].set_title('Price vs Battery Life', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Battery Life (hours)', fontsize=12)
axes[0].set_ylabel('Price (INR)', fontsize=12)

# Price vs Rating
axes[1].scatter(df['rating'], df['price_inr'], alpha=0.5, color='coral')
axes[1].set_title('Price vs Rating', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Rating', fontsize=12)
axes[1].set_ylabel('Price (INR)', fontsize=12)

plt.tight_layout()
plt.show()

In [ ]:
# ANC vs Price
plt.figure(figsize=(10, 6))
df_anc = df[df['active_noise_cancellation'].notna()]
sns.boxplot(data=df_anc, x='active_noise_cancellation', y='price_inr', palette='Set2')
plt.title('Price Distribution: ANC vs Non-ANC', fontsize=16, fontweight='bold')
plt.xlabel('Active Noise Cancellation', fontsize=12)
plt.ylabel('Price (INR)', fontsize=12)
plt.tight_layout()
plt.show()

print("\nANC Price Comparison:")
print(df.groupby('active_noise_cancellation')['price_inr'].agg(['count', 'mean', 'median']))

## 7. Correlation Analysis

In [ ]:
# Select numerical features for correlation
numerical_features = ['price_inr', 'rating', 'review_count', 'battery_life_hrs', 
                      'driver_size_mm', 'bluetooth_version', 'mic_count']
corr_df = df[numerical_features].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_df, annot=True, fmt='.2f', cmap='coolwarm', center=0, 
            square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Feature Correlation Heatmap', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nCorrelation with Price:")
print(corr_df['price_inr'].sort_values(ascending=False))

## 8. Data Cleaning & Preprocessing

In [ ]:
# Create a copy for processing
df_clean = df.copy()

print(f"Original dataset size: {len(df_clean)}")

# 1. Remove rows with missing price or brand (critical features)
df_clean = df_clean.dropna(subset=['price_inr', 'brand'])
print(f"After removing missing price/brand: {len(df_clean)}")

# 2. Remove duplicates based on product_name
df_clean = df_clean.drop_duplicates(subset=['product_name'], keep='first')
print(f"After removing duplicates: {len(df_clean)}")

# 3. Handle missing values in numerical features
# Fill battery_life with median by category
df_clean['battery_life_hrs'] = df_clean.groupby('category')['battery_life_hrs'].transform(
    lambda x: x.fillna(x.median())
)

# Fill bluetooth_version with mode (most common)
df_clean['bluetooth_version'] = df_clean['bluetooth_version'].fillna(df_clean['bluetooth_version'].mode()[0])

# Fill driver_size with median by category
df_clean['driver_size_mm'] = df_clean.groupby('category')['driver_size_mm'].transform(
    lambda x: x.fillna(x.median())
)

# Fill rating with median
df_clean['rating'] = df_clean['rating'].fillna(df_clean['rating'].median())

# Fill review_count with 0 (no reviews)
df_clean['review_count'] = df_clean['review_count'].fillna(0)

# Fill ANC with 0 (no ANC)
df_clean['active_noise_cancellation'] = df_clean['active_noise_cancellation'].fillna(0)

# Fill mic_count with 2 (standard for most headphones)
df_clean['mic_count'] = df_clean['mic_count'].fillna(2)

print(f"\nMissing values after cleaning:")
print(df_clean[numerical_features].isnull().sum())

## 9. Feature Engineering

In [ ]:
# 1. has_noise_cancellation (binary)
df_clean['has_noise_cancellation'] = (df_clean['active_noise_cancellation'] == 1).astype(int)

# 2. price_per_hour (value metric)
df_clean['price_per_hour'] = df_clean['price_inr'] / (df_clean['battery_life_hrs'] + 1)  # +1 to avoid division by zero

# 3. brand_tier (based on average price)
brand_avg_price = df_clean.groupby('brand')['price_inr'].mean()
def categorize_brand(brand):
    if brand not in brand_avg_price:
        return 'mid'
    avg_price = brand_avg_price[brand]
    if avg_price >= 10000:
        return 'premium'
    elif avg_price >= 3000:
        return 'mid'
    else:
        return 'budget'

df_clean['brand_tier'] = df_clean['brand'].apply(categorize_brand)

# 4. bluetooth_major_version (extract major version)
df_clean['bluetooth_major_version'] = df_clean['bluetooth_version'].apply(lambda x: int(x) if pd.notna(x) else 5)

# 5. high_rating (4.0+)
df_clean['high_rating'] = (df_clean['rating'] >= 4.0).astype(int)

# 6. has_ipx_rating (water resistance)
df_clean['has_ipx_rating'] = df_clean['ipx_rating'].notna().astype(int)

print("\nNew Features Created:")
print(f"  - has_noise_cancellation: {df_clean['has_noise_cancellation'].sum()} products with ANC")
print(f"  - price_per_hour: Range ₹{df_clean['price_per_hour'].min():.2f} - ₹{df_clean['price_per_hour'].max():.2f}")
print(f"  - brand_tier: {df_clean['brand_tier'].value_counts().to_dict()}")
print(f"  - bluetooth_major_version: {df_clean['bluetooth_major_version'].value_counts().sort_index().to_dict()}")
print(f"  - high_rating: {df_clean['high_rating'].sum()} products with rating ≥ 4.0")
print(f"  - has_ipx_rating: {df_clean['has_ipx_rating'].sum()} products with IPX rating")

## 10. Encode Categorical Features

In [ ]:
# Label encode categorical features
label_encoders = {}

categorical_features = ['category', 'brand', 'brand_tier']

for feature in categorical_features:
    le = LabelEncoder()
    df_clean[f'{feature}_encoded'] = le.fit_transform(df_clean[feature])
    label_encoders[feature] = le
    print(f"\n{feature} encoding:")
    for i, label in enumerate(le.classes_):
        print(f"  {label}: {i}")

## 11. Prepare Features for Modeling

In [ ]:
# Select features for modeling
feature_columns = [
    'category_encoded',
    'brand_encoded',
    'brand_tier_encoded',
    'rating',
    'review_count',
    'battery_life_hrs',
    'driver_size_mm',
    'bluetooth_version',
    'bluetooth_major_version',
    'mic_count',
    'has_noise_cancellation',
    'has_ipx_rating',
    'high_rating',
    'price_per_hour'
]

target_column = 'price_inr'

# Create feature matrix and target vector
X = df_clean[feature_columns]
y = df_clean[target_column]

print(f"Feature matrix shape: {X.shape}")
print(f"Target vector shape: {y.shape}")
print(f"\nFeatures: {feature_columns}")

## 12. Train-Test Split

In [ ]:
# Split data into train and test sets (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set size: {len(X_train)} ({len(X_train)/len(X)*100:.1f}%)")
print(f"Test set size: {len(X_test)} ({len(X_test)/len(X)*100:.1f}%)")
print(f"\nTraining set price range: ₹{y_train.min():.2f} - ₹{y_train.max():.2f}")
print(f"Test set price range: ₹{y_test.min():.2f} - ₹{y_test.max():.2f}")

## 13. Feature Scaling

In [ ]:
# Standardize numerical features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Feature scaling completed.")
print(f"\nScaled training set - Mean: {X_train_scaled.mean():.4f}, Std: {X_train_scaled.std():.4f}")
print(f"Scaled test set - Mean: {X_test_scaled.mean():.4f}, Std: {X_test_scaled.std():.4f}")

## 14. Save Processed Dataset

In [ ]:
# Save the cleaned and processed dataset
output_path = '../data/processed_data.csv'
df_clean.to_csv(output_path, index=False)

print(f"✅ Processed dataset saved to: {output_path}")
print(f"\nFinal dataset shape: {df_clean.shape}")
print(f"Features ready for modeling: {len(feature_columns)}")
print(f"\nDataset completeness:")
completeness = (1 - df_clean[feature_columns].isnull().sum() / len(df_clean)) * 100
print(completeness.sort_values(ascending=False))

## 15. Summary & Next Steps

In [ ]:
print("=" * 80)
print("EDA & DATA PREPARATION SUMMARY")
print("=" * 80)
print(f"\n✅ Dataset loaded: {len(df)} records")
print(f"✅ Data cleaned: {len(df_clean)} records (removed {len(df) - len(df_clean)} duplicates/invalid)")
print(f"✅ Missing values handled: {len(numerical_features)} numerical features imputed")
print(f"✅ Feature engineering: 6 new features created")
print(f"✅ Categorical encoding: {len(categorical_features)} features encoded")
print(f"✅ Train-test split: {len(X_train)}/{len(X_test)} (80/20)")
print(f"✅ Feature scaling: StandardScaler applied")
print(f"✅ Processed dataset saved: {output_path}")
print(f"\n📊 Ready for model training with {len(feature_columns)} features!")
print("\n🎯 Next Steps:")
print("   1. Train multiple regression models (Linear, Ridge, Lasso, RF, GBM)")
print("   2. Evaluate model performance (R², RMSE, MAE)")
print("   3. Select best model with R² ≥ 0.70")
print("   4. Save trained model for deployment")
print("=" * 80)